In [2]:
"""
pdf_reader.py
-------------
Extracts text AND tables from an invoice PDF using spatial layout preservation.

Layout text extraction: Uses `pdfplumber` with `layout=True` to reconstruct
the 2D visual layout of the PDF page from character/word coordinates.
This preserves columns, tables, and spacing so the LLM has exact spatial
awareness.

Dual-stream view: Uses `pypdfium2` (Apache-2.0 / BSD-3-Clause) for sequential
text extraction to provide a column-separated stream view alongside the
pdfplumber layout view. This prevents garbled side-by-side columns
(e.g. interleaving "Tesla" + "Tesla" → "TTeessllaa").

Table extraction: Uses `pdfplumber.extract_tables()` to detect cell grids,
rendered as formatted markdown tables.

OCR fallback: When a PDF is purely a scanned image with no embedded text,
falls back to pypdfium2 rasterization + RapidOCR / PaddleOCR. Every tier of
this fallback chain logs why it was or wasn't used -- a silently-skipped
tier (Docker not installed, an OCR engine failing to import) used to be
invisible; now it shows up in the logs instead of just a quality drop.

License: All PDF libraries used here are permissively licensed:
  - pdfplumber: MIT
  - pypdfium2: Apache-2.0 / BSD-3-Clause (Google PDFium)
  - rapidocr_onnxruntime: Apache-2.0
"""
from __future__ import annotations
from pathlib import Path

import logging
import math
import os
import re
import shutil
import statistics

import pdfplumber
import pypdfium2 as pdfium

logger = logging.getLogger("invoice_extractor")


# Marks a detected column boundary in the 2D layout canvas -- a box-drawing
# character chosen specifically because it never appears in real invoice
# text, unlike "|" which occasionally shows up in part numbers/codes.
_COLUMN_BOUNDARY_MARKER = "│"  # "│"


def _detect_column_boundaries(
    lines: list[list[dict]],
    page: pdfplumber.page.Page,
    char_width_pts: float,
    min_gap_multiplier: float = 4.0,
    min_recurrence_fraction: float = 0.15,
    min_recurrence_count: int = 3,
) -> list[float]:
    """
    Returns x-positions (in points) that are confirmed table-column
    boundaries on this page, from two signals:

    1. Recurring wide gaps: a horizontal gap between two adjacent words on
       a line only counts as a real column boundary if a gap shows up at
       roughly the *same x-position across multiple lines* -- that's the
       actual signature of a table column, as opposed to one line just
       happening to have a wide gap (e.g. trailing spaces after a label
       like "Total:      "). A single wide gap on one line is ignored.
    2. Ruled vertical lines the document itself drew (page.lines /
       page.rects) -- when there's a real column divider printed on the
       page, trust it directly; it doesn't need to recur across lines.
    """
    cluster_tolerance_pts = char_width_pts * 2
    min_gap_pts = char_width_pts * min_gap_multiplier

    # 1. Collect wide-gap midpoints from inter-word spacing on each line.
    gap_midpoints: list[float] = []
    for line in lines:
        for prev_w, next_w in zip(line, line[1:]):
            gap = next_w["x0"] - prev_w["x1"]
            if gap >= min_gap_pts:
                gap_midpoints.append((prev_w["x1"] + next_w["x0"]) / 2.0)

    gap_midpoints.sort()
    bands: list[list[float]] = []
    for x in gap_midpoints:
        if bands and x - bands[-1][-1] <= cluster_tolerance_pts:
            bands[-1].append(x)
        else:
            bands.append([x])

    min_recurrence = max(min_recurrence_count, int(len(lines) * min_recurrence_fraction))
    confirmed = [sum(band) / len(band) for band in bands if len(band) >= min_recurrence]

    # 2. Ruled vertical lines -- a tall, thin line/rect is a drawn column
    # divider. Always trusted, no recurrence needed.
    min_rule_height = page.height * 0.05
    for line_obj in page.lines or []:
        if abs(line_obj["x0"] - line_obj["x1"]) <= 1.0 and (line_obj["bottom"] - line_obj["top"]) >= min_rule_height:
            confirmed.append(line_obj["x0"])
    for rect in page.rects or []:
        if rect["width"] <= 1.5 and rect["height"] >= min_rule_height:
            confirmed.append(rect["x0"])

    return sorted(set(round(x, 1) for x in confirmed))


def _gap_crosses_boundary(gap_start_pts: float, gap_end_pts: float, boundaries: list[float]) -> bool:
    return any(gap_start_pts <= b <= gap_end_pts for b in boundaries)


def extract_page_layout_canvas(
    page: pdfplumber.page.Page,
    char_width_pts: float | None = None,
    v_tolerance_factor: float = 0.35,
) -> str:
    """
    Extracts complete text from a PDF page onto a 2D spatial canvas.
    Preserves exact linearity, columns, and spacing without word collisions.

    Confirmed table-column boundaries (see _detect_column_boundaries) get
    an explicit marker (_COLUMN_BOUNDARY_MARKER) instead of plain
    whitespace, wherever a line's own gap actually crosses one -- so a
    model reading the flattened text can tell "two separate columns with
    a wide gap between them" apart from "one long run-on sentence with
    incidental extra spacing," which plain whitespace can't distinguish.
    """
    words = page.extract_words(x_tolerance=1.5, y_tolerance=1.5)
    if not words:
        return ""

    # 1. Adaptive character width from page font metrics
    if char_width_pts is None:
        if page.chars:
            valid_widths = [c['width'] for c in page.chars if c.get('text', '').strip()]
            char_width_pts = round(statistics.median(valid_widths), 2) if valid_widths else 4.0
        else:
            char_width_pts = 4.0

    # 2. Adaptive vertical tolerance based on font heights
    font_heights = [w['bottom'] - w['top'] for w in words]
    med_height = statistics.median(font_heights) if font_heights else 8.0
    v_tolerance = max(2.0, med_height * v_tolerance_factor)

    # 3. Sort words top-to-bottom, left-to-right
    sorted_words = sorted(words, key=lambda w: (w['top'], w['x0']))

    # 4. Group into horizontal lines
    lines: list[list[dict]] = []
    curr_line: list[dict] = []
    curr_center: float | None = None

    for w in sorted_words:
        w_center = (w['top'] + w['bottom']) / 2.0
        if curr_center is None:
            curr_center = w_center
            curr_line.append(w)
        elif abs(w_center - curr_center) <= v_tolerance:
            curr_line.append(w)
            curr_center = sum((x['top'] + x['bottom']) / 2.0 for x in curr_line) / len(curr_line)
        else:
            lines.append(sorted(curr_line, key=lambda x: x['x0']))
            curr_line = [w]
            curr_center = w_center

    if curr_line:
        lines.append(sorted(curr_line, key=lambda x: x['x0']))

    # 4b. Detect recurring/ruled column boundaries across the whole page
    # before rendering -- rendering needs to know about them per-line.
    column_boundaries = _detect_column_boundaries(lines, page, char_width_pts)

    # 5. Render onto 2D character canvas with collision prevention
    total_cols = int(math.ceil(page.width / char_width_pts)) + 20
    output_rows: list[str] = []

    for line in lines:
        row = [' '] * total_cols
        curr_col = 0
        for idx, w in enumerate(line):
            target_col = int(round(w['x0'] / char_width_pts))
            # Collision prevention: advance if target is already occupied
            col = max(target_col, curr_col + 1 if curr_col > 0 else target_col)

            # If THIS line's own gap from the previous word is itself wide
            # (not just a normal single space) AND that gap crosses a
            # confirmed column boundary, mark it explicitly instead of
            # leaving it as plain whitespace. Both conditions matter: a
            # confirmed boundary is a single clustered x-position, and
            # checking overlap alone would flag any narrow, ordinary
            # inter-word gap that happens to straddle that x by
            # coincidence (e.g. "Doddanakundi Industrial" -- a normal
            # ~2pt space -- got wrongly split this way before this check
            # was added, just because a table's column gap elsewhere on
            # the page averaged to an x-position that fell inside it).
            if idx > 0 and column_boundaries:
                prev_w = line[idx - 1]
                this_gap = w["x0"] - prev_w["x1"]
                if this_gap >= char_width_pts * 1.5 and _gap_crosses_boundary(prev_w["x1"], w["x0"], column_boundaries):
                    delim_col = (curr_col + col) // 2
                    if curr_col <= delim_col < len(row):
                        row[delim_col] = _COLUMN_BOUNDARY_MARKER

            w_text = w['text']
            for i, ch in enumerate(w_text):
                pos = col + i
                if pos < len(row):
                    row[pos] = ch
            curr_col = col + len(w_text)

        output_rows.append(''.join(row).rstrip())

    return '\n'.join(output_rows)


def extract_full_pdf(pdf_path: str | Path, char_width_pts: float | None = None) -> str:
    """
    Extracts the ENTIRE PDF document (all pages) preserving complete wordings,
    linearity, column separation, and spacing.
    Only includes pages that contain actual text characters (scanned pages with 0 chars are skipped).
    """
    pdf_path = Path(pdf_path)
    extracted_pages: list[str] = []

    with pdfplumber.open(str(pdf_path)) as pdf:
        num_pages = len(pdf.pages)
        for idx, page in enumerate(pdf.pages, start=1):
            page_text = extract_page_layout_canvas(page, char_width_pts=char_width_pts)
            if page_text and page_text.strip():
                header = f"\n{'=' * 35} PAGE {idx} OF {num_pages} ({pdf_path.name}) {'=' * 35}\n"
                extracted_pages.append(header + page_text)

    return "\n".join(extracted_pages).strip()


# Regex for genuine words containing at least one alphanumeric character --
# shared by every per-page text-density check below.
_WORD_PATTERN = re.compile(r"[a-zA-Z0-9]")


def _page_text_stats(pdf_path: str | Path, min_word_len: int = 2) -> list[dict[str, int]]:
    """
    Opens the PDF ONCE and returns per-page text-density stats:
        [{"chars": int, "valid_words": int, "single_char_words": int}, ...]

    Both is_scanned_pdf() and the --force-ocr/--skip-text decision need
    per-page word/char counts -- this is the one place that computes them,
    so neither re-parses the document or keeps a second definition of
    "what counts as a real word on a page."
    """
    stats: list[dict[str, int]] = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            page_text = page.extract_text() or ""

            valid_words = 0
            single_char_words = 0
            for w in words:
                t = w.get("text", "").strip()
                if _WORD_PATTERN.search(t):
                    if len(t) >= min_word_len:
                        valid_words += 1
                    else:
                        # Catch single-character vertical letter noise (like D \n R \n E)
                        single_char_words += 1

            stats.append({
                "chars": len(page_text.strip()),
                "valid_words": valid_words,
                "single_char_words": single_char_words,
            })
    return stats


def is_scanned_pdf(
    pdf_path: str | Path,
    min_words_per_page: int = 25,
    min_word_len: int = 2,
    min_avg_chars_per_page: int = 150,
) -> bool:
    """
    Strictly evaluates whether a PDF is a scanned image (or has garbage OCR
    layers) versus a true readable digital PDF, using PER-PAGE density
    checks rather than a single whole-document character count.

    That distinction matters: extract_text()'s old gate was just "does the
    whole PDF have >= 40 characters of embedded text anywhere" -- a
    multi-page scanned invoice with nothing but a letterhead or a single
    stamped cover line clears that easily while every other page is a pure
    image, so the whole document got (wrongly) treated as fully digital.
    Checking density page-by-page catches that.

    Returns:
        True  -> PDF is SCANNED (or corrupted/sparse text layer) -> route to OCR.
        False -> PDF is a true digital PDF with dense, readable text.
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        return True

    try:
        page_stats = _page_text_stats(pdf_path, min_word_len=min_word_len)
    except Exception as e:
        logger.debug(
            "%s: is_scanned_pdf check failed (%s), treating as scanned to trigger OCR",
            pdf_path, e,
        )
        return True

    total_pages = len(page_stats)
    if total_pages == 0:
        return True

    total_chars = sum(p["chars"] for p in page_stats)
    total_valid_words = sum(p["valid_words"] for p in page_stats)
    single_char_word_count = sum(p["single_char_words"] for p in page_stats)
    pages_with_dense_text = sum(1 for p in page_stats if p["valid_words"] >= min_words_per_page)

    # 1. Average characters per page (commercial invoices run ~300-2000 chars/page)
    avg_chars_per_page = total_chars / total_pages
    if avg_chars_per_page < min_avg_chars_per_page:
        logger.debug(
            "%s: avg %.0f chars/page < %d -> scanned",
            pdf_path, avg_chars_per_page, min_avg_chars_per_page,
        )
        return True

    # 2. Average real words per page
    avg_words_per_page = total_valid_words / total_pages
    if avg_words_per_page < min_words_per_page:
        logger.debug(
            "%s: avg %.1f words/page < %d -> scanned",
            pdf_path, avg_words_per_page, min_words_per_page,
        )
        return True

    # 3. Vertical fragmentation / garbage letter noise (rotated or corrupted text layer)
    if single_char_word_count > (total_valid_words * 0.8):
        logger.debug(
            "%s: %d single-char word fragments vs %d real words -> scanned/corrupted",
            pdf_path, single_char_word_count, total_valid_words,
        )
        return True

    # 4. In multi-page PDFs, at least half the pages should have meaningful content
    if total_pages > 1 and pages_with_dense_text < (total_pages / 2):
        logger.debug(
            "%s: only %d/%d pages have dense text -> scanned",
            pdf_path, pages_with_dense_text, total_pages,
        )
        return True

    return False


def _has_partial_text_pages(
    pdf_path: str | Path,
    min_words_per_page: int = 25,
    min_word_len: int = 2,
) -> bool:
    """
    True if any page has a text layer that isn't dense/real enough --
    the specific case OCRmyPDF's --skip-text mishandles.

    The check is `chars > 0 and valid_words < min_words_per_page`, NOT
    `0 < valid_words < min_words_per_page` -- those are not the same thing,
    and the difference matters. A real page found in production: 4674 raw
    characters (`page.extract_text()`), but 0 "valid words" by our
    alphanumeric-word-of-length>=2 definition -- i.e. pdfplumber's word
    extractor found nothing it considered a real word, but there plainly
    *was* a text layer there (garbled/corrupted encoding, not a blank
    page). `valid_words == 0` alone made this look like a pure image page
    ("nothing there for --skip-text to skip"), so --skip-text was
    auto-selected -- and it wrongly skipped the page anyway, because
    OCRmyPDF's own "does this page already have text" check isn't
    word-quality-aware either; it just sees *a* text layer and bails.
    Forcing OCR on that exact file recovered the real content.

    So: a page only counts as "nothing here, --skip-text is safe" when it
    has ZERO extracted characters at all, not merely zero recognizable
    words. Any non-zero character count below the dense-page word bar is
    "partial" and should force OCR.

    Still reuses the exact same per-page stats and the same
    min_words_per_page bar as is_scanned_pdf() -- just testing `chars`
    instead of `valid_words` for the "is there truly nothing here" leg.
    """
    try:
        page_stats = _page_text_stats(pdf_path, min_word_len=min_word_len)
    except Exception as e:
        logger.debug(
            "%s: partial-text-page check failed (%s), defaulting to --force-ocr to be safe",
            pdf_path, e,
        )
        return True

    return any(p["chars"] > 0 and p["valid_words"] < min_words_per_page for p in page_stats)


# Global OCR engine cache so neural network models load once into memory,
# saving 5-10 seconds on every subsequent PDF extraction.
_OCR_ENGINE = None
_OCR_ENGINE_TYPE = None


def _get_ocr_engine():
    """Initializes and caches the fastest available pure-Python OCR engine."""
    global _OCR_ENGINE, _OCR_ENGINE_TYPE
    if _OCR_ENGINE is not None:
        return _OCR_ENGINE, _OCR_ENGINE_TYPE

    # Disable buggy experimental Paddle PIR/OneDNN compiler on Windows CPU
    try:
        import paddle
        paddle.set_flags({"FLAGS_enable_pir_api": 0})
    except Exception as e:
        logger.debug("Paddle PIR flag tweak skipped (paddle not installed or old version): %s", e)

    # 1. Try RapidOCR first (rock-solid ONNX runtime on Windows CPU, ~1s per page)
    try:
        from rapidocr_onnxruntime import RapidOCR
        _OCR_ENGINE = RapidOCR()
        _OCR_ENGINE_TYPE = "rapidocr"
        logger.info("OCR engine: using RapidOCR")
        return _OCR_ENGINE, _OCR_ENGINE_TYPE
    except Exception as e:
        logger.debug("RapidOCR unavailable, trying PaddleOCR next: %s", e)

    # 2. Try standard PaddleOCR with OneDNN/MKLDNN disabled to prevent Windows C++ crashes
    try:
        from paddleocr import PaddleOCR
        try:
            _OCR_ENGINE = PaddleOCR(use_angle_cls=True, lang="en", enable_mkldnn=False)
        except Exception:
            try:
                _OCR_ENGINE = PaddleOCR(lang="en", enable_mkldnn=False)
            except Exception:
                _OCR_ENGINE = PaddleOCR(lang="en")
        _OCR_ENGINE_TYPE = "paddleocr"
        logger.info("OCR engine: using PaddleOCR")
        return _OCR_ENGINE, _OCR_ENGINE_TYPE
    except Exception as e:
        logger.debug("PaddleOCR unavailable, trying PaddleOCRVL next: %s", e)

    # 3. Try PaddleOCRVL (heavy VL pipeline)
    try:
        from paddleocr import PaddleOCRVL
        try:
            _OCR_ENGINE = PaddleOCRVL(pipeline_version="v1.5")
        except Exception:
            _OCR_ENGINE = PaddleOCRVL()
        _OCR_ENGINE_TYPE = "paddleocr_vl"
        logger.info("OCR engine: using PaddleOCRVL")
        return _OCR_ENGINE, _OCR_ENGINE_TYPE
    except Exception as e:
        logger.debug("PaddleOCRVL unavailable: %s", e)

    raise ImportError(
        "Scanned image PDF detected, but PaddleOCR/RapidOCR is not ready in Python.\n"
        "Please run: pip install rapidocr_onnxruntime\n"
        "or: pip install paddleocr"
    )


def _ocr(pdf_path: str | Path, dpi: int = 200) -> tuple[str, str, str, float]:
    """OCR fallback for scanned / image-only PDFs.
    Uses cached pure-Python PaddleOCR / RapidOCR on pypdfium2-rendered page images.
    Returns (full_ocr_text, full_ocr_tables, engine_type, ocr_time_seconds).
    """
    import tempfile
    import time

    start_ocr = time.perf_counter()
    engine, engine_type = _get_ocr_engine()

    ocr_pages: list[str] = []
    ocr_tables: list[str] = []

    scale = dpi / 72  # pypdfium2 uses scale factor (72 DPI base)
    doc = pdfium.PdfDocument(str(pdf_path))
    for page_idx in range(len(doc)):
        page = doc[page_idx]
        bitmap = page.render(scale=scale)
        pil_image = bitmap.to_pil()

        # Save page image temporarily for OCR engine
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_img:
            pil_image.save(tmp_img, format="PNG")
            tmp_img_path = tmp_img.name

        lines_extracted: list[str] = []
        try:
            # Option A: RapidOCR (Fastest ONNX, ~1s per page)
            if engine_type == "rapidocr":
                with open(tmp_img_path, "rb") as f:
                    img_bytes = f.read()
                result, _ = engine(img_bytes)
                if result:
                    for line in result:
                        if line and len(line) > 1:
                            lines_extracted.append(str(line[1]))

            # Option B: Standard PaddleOCR (~2s per page)
            elif engine_type == "paddleocr":
                result = None
                try:
                    result = engine.ocr(tmp_img_path)
                except Exception as pe:
                    logger.warning(
                        "PaddleOCR failed on page %d (%s), falling back to RapidOCR for this page",
                        page_idx + 1, pe,
                    )
                    # Fallback to RapidOCR if Paddle OneDNN crashes on Windows
                    try:
                        from rapidocr_onnxruntime import RapidOCR
                        r_engine = RapidOCR()
                        with open(tmp_img_path, "rb") as f:
                            img_bytes = f.read()
                        r_res, _ = r_engine(img_bytes)
                        if r_res:
                            for line in r_res:
                                if line and len(line) > 1:
                                    lines_extracted.append(str(line[1]))
                    except Exception as e2:
                        logger.warning(
                            "RapidOCR page-level fallback also failed on page %d: %s",
                            page_idx + 1, e2,
                        )

                if result:
                    for page_res in result:
                        if hasattr(page_res, "text") and page_res.text:
                            lines_extracted.append(str(page_res.text))
                        elif isinstance(page_res, dict) and "text" in page_res:
                            lines_extracted.append(str(page_res["text"]))
                        elif isinstance(page_res, (list, tuple)):
                            for line in page_res:
                                if isinstance(line, (list, tuple)) and len(line) > 1 and isinstance(line[1], (list, tuple)) and len(line[1]) > 0:
                                    lines_extracted.append(str(line[1][0]))
                                elif isinstance(line, str):
                                    lines_extracted.append(line)
                        elif hasattr(page_res, "__str__"):
                            lines_extracted.append(str(page_res))

            # Option C: PaddleOCRVL (Heavy VL pipeline)
            elif engine_type == "paddleocr_vl":
                output = engine.predict(tmp_img_path)
                for res in output:
                    if hasattr(res, "markdown") and res.markdown:
                        lines_extracted.append(str(res.markdown))
                    elif hasattr(res, "text") and res.text:
                        lines_extracted.append(str(res.text))
                    elif hasattr(res, "json") and res.json:
                        for item in res.json.get("layout_elements", []):
                            if "text" in item:
                                lines_extracted.append(item["text"])
                    elif isinstance(res, dict) and "text" in res:
                        lines_extracted.append(res["text"])
                    else:
                        lines_extracted.append(str(res))

        finally:
            if os.path.exists(tmp_img_path):
                try:
                    os.unlink(tmp_img_path)
                except Exception:
                    pass  # best-effort temp cleanup, not worth logging

        page.close()

        if lines_extracted:
            ocr_pages.append(f"--- Page {page_idx + 1} (OCR) ---\n" + "\n".join(lines_extracted))

    doc.close()

    ocr_time_seconds = round(time.perf_counter() - start_ocr, 2)
    full_ocr_text = "\n\n".join(ocr_pages).strip()
    full_ocr_tables = "\n\n".join(ocr_tables).strip()
    return full_ocr_text, full_ocr_tables, engine_type, ocr_time_seconds


# normalize_pdf_rotation() removed: pypdf's page.rotate() only overwrites the
# /Rotate dictionary entry, it never transforms the content stream's own
# coordinates. pdfplumber already applies /Rotate correctly on its own when
# reporting word/char positions, so zeroing it out post-hoc left pdfplumber
# reading the SAME raw (still-rotated) coordinates as if no rotation were
# needed -- silently scrambling every rotated page's line order. Confirmed
# on a real 270-degree-rotated invoice: extract_full_pdf() on the untouched
# file produced clean text; routing it through this function first produced
# character soup. pdfplumber needs no help here.


def _ocr_with_docker(
    input_path: str | Path, dpi: int = 300, force_ocr: bool = False
) -> tuple[Path | None, float]:
    """
    Uses the official OCRmyPDF Docker container (jbarlow83/ocrmypdf-alpine)
    to generate a searchable PDF with exact embedded text coordinates.
    Includes --rotate-pages to automatically detect and correct scanned page orientation using Tesseract OSD.

    By default passes --skip-text, which skips OCR on any page that
    already has *some* text layer -- fast, but it means a page with only
    a sliver of pre-existing garbage text (a stray watermark, a corrupted
    text remnant) gets skipped instead of OCR'd, leaving that page's real
    content unrecovered. `force_ocr=True` passes --force-ocr instead,
    which rasterizes and re-OCRs every page unconditionally -- slower, but
    is the fix when --skip-text is the reason a scan came back empty.

    Returns (Path_to_pdf_or_None, ocr_time_seconds).
    """
    import subprocess
    import tempfile
    import time

    in_file = Path(input_path)
    if not in_file.exists():
        return None, 0.0

    # Fast pre-check: don't attempt (and don't wait out the subprocess
    # timeout for) a Docker call when Docker itself isn't even installed.
    if shutil.which("docker") is None:
        logger.info("Docker OCR fallback skipped: 'docker' not found on PATH")
        return None, 0.0

    start_docker = time.perf_counter()
    tmp_out = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
    tmp_out.close()

    cmd = [
        "docker", "run", "--rm", "-i",
        "jbarlow83/ocrmypdf-alpine",
        "--image-dpi", str(dpi),
        "--rotate-pages",
        "--force-ocr" if force_ocr else "--skip-text",
        "-", "-"
    ]

    try:
        with open(in_file, "rb") as infile, open(tmp_out.name, "wb") as outfile:
            proc = subprocess.run(
                cmd,
                stdin=infile,
                stdout=outfile,
                stderr=subprocess.PIPE,
                timeout=180,
            )
        elapsed = round(time.perf_counter() - start_docker, 2)
        if proc.returncode == 0 and os.path.getsize(tmp_out.name) > 0:
            logger.info("Docker OCRmyPDF succeeded in %.2fs", elapsed)
            return Path(tmp_out.name), elapsed
        else:
            logger.info(
                "Docker OCRmyPDF produced no usable output (returncode=%s): %s",
                proc.returncode, proc.stderr.decode(errors="ignore")[:300],
            )
            if os.path.exists(tmp_out.name):
                try:
                    os.unlink(tmp_out.name)
                except Exception:
                    pass
            return None, 0.0
    except subprocess.TimeoutExpired:
        logger.info("Docker OCRmyPDF timed out after 180s, falling back to Python OCR")
        if os.path.exists(tmp_out.name):
            try:
                os.unlink(tmp_out.name)
            except Exception:
                pass
        return None, 0.0
    except Exception as e:
        logger.info("Docker OCR fallback skipped due to error: %s", e)
        if os.path.exists(tmp_out.name):
            try:
                os.unlink(tmp_out.name)
            except Exception:
                pass
        return None, 0.0


def visualize_document_layout(pdf_path: str | Path, page_num: int = 0, dpi: int = 120):
    """
    Visualizes bounding boxes on the PDF page for debugging layout and alignment.
    """
    import matplotlib.pyplot as plt

    pdf_path = Path(pdf_path)
    with pdfplumber.open(str(pdf_path)) as pdf:
        if page_num >= len(pdf.pages):
            raise IndexError(f"Page {page_num} out of range (PDF has {len(pdf.pages)} pages).")
        page = pdf.pages[page_num]
        page_img = page.to_image(resolution=dpi)
        words = page.extract_words()
        annotated = page_img.draw_rects(words, stroke="red", stroke_width=1, fill=None)

        fig, ax = plt.subplots(figsize=(10, 10 * (page.height / page.width)))
        ax.imshow(annotated.annotated)
        ax.axis('off')
        ax.set_title(f"Visual Debug: {pdf_path.name} - Page {page_num + 1} ({len(words)} words)", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()


def extract_text(
    pdf_path: str | Path,
    min_chars_for_text_layer: int = 40,
    force_ocr: bool | None = None,
) -> tuple[str, str, str, float]:
    """
    Returns (text, tables_text, method, ocr_time_seconds).

    - text: High-precision 2D layout canvas preserving exact vertical linearity,
            horizontal column separation, and word collision prevention.
    - tables_text: Empty string (tables are preserved directly within the 2D layout canvas).
    - method: "2d_layout_canvas", "ocrmypdf_docker", or f"ocr_{engine_type}"
    - ocr_time_seconds: time spent running OCR (0.0 for digital PDFs with embedded text layer)
    - force_ocr: controls both whether the digital text-layer path is even
      attempted, and which flag the Docker OCR pass uses:
        * None (default) -- auto. Try the digital text layer first (unless
          is_scanned_pdf() already says this is scanned); if OCR is needed,
          auto-decide --skip-text vs --force-ocr per file based on whether
          any page has a partial text layer (see _has_partial_text_pages()).
        * True  -- always skip the digital-text check and OCR every page,
          using OCRmyPDF's --force-ocr.
        * False -- always try the digital text layer first, and if OCR is
          needed, always use --skip-text (never auto-upgrade to --force-ocr).
    """
    skip_digital_path = force_ocr is True

    if not skip_digital_path:
        # 1. Fast path: High-Precision 2D Layout Canvas from embedded text
        # layer -- but only attempt it if is_scanned_pdf() agrees this is a
        # genuine digital PDF. A single sparse page (letterhead, a stamped
        # cover line) could otherwise clear min_chars_for_text_layer on its
        # own while every other page is a pure image.
        already_known_scanned = force_ocr is None and is_scanned_pdf(pdf_path)
        if not already_known_scanned:
            # pdfplumber already applies a page's /Rotate transform itself
            # when reporting word/char coordinates -- no separate
            # normalization step needed (see removal note above).
            text = extract_full_pdf(pdf_path)
            if len(text) >= min_chars_for_text_layer:
                return text, "", "2d_layout_canvas", 0.0
            logger.info(
                "Digital text layer in %s was too sparse (<%d chars), falling back to OCR",
                pdf_path, min_chars_for_text_layer,
            )
        else:
            logger.info("%s classified as scanned (sparse/garbage text layer), routing straight to OCR", pdf_path)
    else:
        logger.info("force_ocr=True requested for %s, skipping the digital text-layer check", pdf_path)

    # 2. Decide --skip-text vs --force-ocr for the Docker OCR pass.
    if force_ocr is None:
        use_force_flag = _has_partial_text_pages(pdf_path)
        logger.info(
            "%s: auto-selected %s for the Docker OCR pass",
            pdf_path, "--force-ocr" if use_force_flag else "--skip-text",
        )
    else:
        use_force_flag = force_ocr

    # 3. Scanned PDF / image: Try OCRmyPDF Docker first (with --rotate-pages for auto-orientation)
    docker_pdf, docker_time = _ocr_with_docker(pdf_path, force_ocr=use_force_flag)
    if docker_pdf is not None:
        try:
            d_text = extract_full_pdf(docker_pdf)
            if len(d_text) >= min_chars_for_text_layer:
                return d_text, "", "ocrmypdf_docker", docker_time
        finally:
            if os.path.exists(docker_pdf):
                try:
                    os.unlink(docker_pdf)
                except Exception:
                    pass

    # 4. Fallback to Python OCR (RapidOCR / PaddleOCR / PaddleOCR-VL)
    ocr_text, ocr_tables, engine_type, ocr_time = _ocr(pdf_path)
    return ocr_text, ocr_tables, f"ocr_{engine_type}", ocr_time


# if __name__ == "__main__":
#     import sys
#     t, tabs, m, ot = extract_text(sys.argv[1])
#     print(f"--- extracted via {m} ({len(t)} chars, ocr_time: {ot}s) ---")
#     print(t[:2000])


In [3]:
import sys
t, tabs, m, ot = extract_text(r"D:\prashantproject\FILES\EXPORT\VIBRACOUSTIC MUM SEA EXP 1\VIBRACOUSTIC MUM SEA EXP\PB1126000361 INPL.pdf")
print(f"--- extracted via {m} ({len(t)} chars, ocr_time: {ot}s) ---")
print(t[:10000])

--- extracted via 2d_layout_canvas (15059 chars, ocr_time: 0.0s) ---
=================================== PAGE 1 OF 2 (PB1126000361 INPL.pdf) ===================================
                    VIBRACOUSTIC              INDIA    PRIVATE       LIMITED                                                     CIN   U34300PB1997PTC039144
                                                                                                                                      -
                                                                                       INVOICE
                    Exporter:                                               INVOICE  NO.        DATE            IEC NO.           PAN NO.        SAP  DELIVERY  NO.
                    VIBRACOUSTIC   INDIA PRIVATE LIMITED                    PB1126000361      21-08-2026      0597073619       AABCS7623P             44486502
                    VILLAGE BARI MARUALI  AND RAMGARH                     GSTIN ID: 03AABCS7623P1Z3
            

In [2]:
t, tabs, m, ot = extract_text(r"D:\prashantproject\FILES\EXPORT\Globatronix (Bombay) MUM EXP 1\Globatronix (Bombay) MUM AIR EXP\GBXTDY-392026 INV.pdf")